In [ ]:
import bisect
import csv
import concurrent.futures
import datetime
import faiss
import glob
import gc
import json
import math
import multiprocessing
import os
import pickle
import random
import re
import string
import subprocess
import time
import torch
import uuid
import yaml

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import multiprocessing as mp
import polars as pl
import torch.nn as nn
import torch.optim as optim

from bisect import bisect_left
from collections import defaultdict, Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timedelta
from tqdm import tqdm
from pathlib import Path
from multiprocessing import Pool, cpu_count
from functools import partial
from typing import Set, Tuple, Dict

In [ ]:
# Run this notebook from the repository root.
PROJECT_ROOT = Path.cwd().resolve()

INPUT_ROOT = PROJECT_ROOT / "input"
OUTPUT_ROOT = PROJECT_ROOT / "output"

ALIBABA_INPUT_DIR = INPUT_ROOT / "alibaba_disk"
ALIBABA_SOURCE_DIR = ALIBABA_INPUT_DIR / "original_csv"

SMART_ATTRIBUTE_PATH = INPUT_ROOT / "smart_attributes.csv"
NAN_COLUMNS_PATH = ALIBABA_INPUT_DIR / "completely_nan_columns.txt"
FAULT_TAG_PATH = ALIBABA_INPUT_DIR / "fault_tag_data.csv"

# Alibaba Disk model ID to process. Use 1 or 2.
MODEL_VALUE = 1
MODEL_NAME_MAP = {
    1: "model-one",
    2: "model-two",
}
DISK_MODEL_ID = MODEL_NAME_MAP.get(MODEL_VALUE, f"model-{MODEL_VALUE}")

OUTPUT_DATA_DIR = OUTPUT_ROOT / "data" / DISK_MODEL_ID
OUTPUT_TEXTUAL_DIR = OUTPUT_DATA_DIR / "monthly_textual"
OUTPUT_MODEL_DIR = OUTPUT_ROOT / "models" / DISK_MODEL_ID
OUTPUT_RESULT_PATH = OUTPUT_ROOT / "results" / f"{DISK_MODEL_ID}.csv"

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TEXTUAL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Alibaba source dir : {ALIBABA_SOURCE_DIR}")
print(f"Output data dir    : {OUTPUT_DATA_DIR}")
print(f"Disk model ID      : {DISK_MODEL_ID}")

## Load Metadata and Raw Data

In [ ]:
df_smart_des = pd.read_csv(SMART_ATTRIBUTE_PATH)
print(df_smart_des.columns)

In [ ]:
with open(NAN_COLUMNS_PATH, encoding="utf-8") as file:
    nan_features = [line.strip() for line in file if line.strip()]

files = sorted(ALIBABA_SOURCE_DIR.glob("smartlog_data_*.csv"))
if not files:
    raise FileNotFoundError(f"No Alibaba Disk SMART log files found under: {ALIBABA_SOURCE_DIR}")

temp_df = pd.read_csv(files[0], nrows=3)
feature_columns = [
    col for col in temp_df.columns
    if col.startswith("smart") and col.endswith("raw") and col not in nan_features
]

basic_columns = [
    "serial_number",
    "manufacturer",
    "model",
    "dt",
]

del temp_df

print(f"Found {len(files)} SMART log files.")
print(f"Number of SMART raw columns used: {len(feature_columns)}")
print(f"Basic columns: {basic_columns}")

In [ ]:
lf = (
    pl.scan_csv(
        [str(path) for path in files],
        try_parse_dates=True,
        infer_schema_length=100_000,
        low_memory=True,
        rechunk=False,
    )
    .select(basic_columns + feature_columns)
)

df_pl = lf.collect(streaming=True)
df = df_pl.to_pandas(use_pyarrow_extension_array=True)

df["datetime"] = pd.to_datetime(df["dt"], format="%Y%m%d", errors="coerce").dt.strftime("%Y-%m-%d")
df = df.drop(columns=["dt"]).reset_index(drop=True)

# Align Alibaba SMART column names to the common schema:
# e.g., smart_5raw -> smart_5_raw
df.columns = df.columns.str.replace(r"(\d+)raw$", r"\1_raw", regex=True)

print(df.shape)
df.head(3)

## Build Failure Labels from Data

In [ ]:
failure_df = pd.read_csv(FAULT_TAG_PATH)

failure_df = (
    failure_df
    .groupby(["serial_number", "fault_time", "manufacturer", "model"], as_index=False)
    .agg({"tag": lambda x: sorted(set(x))})
    .rename(columns={"tag": "tags_list"})
)

df["failure"] = 0

df["key"] = (
    df["serial_number"].astype("string")
    + "_"
    + df["datetime"].astype("string")
    + "_"
    + df["model"].astype("string")
)

failure_df["key"] = (
    failure_df["serial_number"].astype("string")
    + "_"
    + failure_df["fault_time"].astype("string")
    + "_"
    + failure_df["model"].astype("string")
)

df.loc[df["key"].isin(failure_df["key"]), "failure"] = 1
df = df.drop(columns=["key"])

print(f"Matched failure rows: {int(df['failure'].sum()):,}")

In [ ]:
df = df[df["model"] == MODEL_VALUE].copy()

failed_serials = df[df["failure"] == 1]["serial_number"].unique()
all_serials = df["serial_number"].unique()
healthy_serials = np.setdiff1d(all_serials, failed_serials)

# Keep all failed disks and sample a portion of never-failed healthy disks.
HEALTHY_SAMPLE_RATIO = 0.50
rng = np.random.default_rng(seed=42)

n_sample = int(len(healthy_serials) * HEALTHY_SAMPLE_RATIO)
sampled_healthy = rng.choice(healthy_serials, size=n_sample, replace=False) if n_sample > 0 else []

keep_serials = np.concatenate([failed_serials, sampled_healthy])
df = df[df["serial_number"].isin(keep_serials)].copy()

df = df.rename(columns={"datetime": "date"}).reset_index(drop=True)

print(f"Target model value: {MODEL_VALUE}")
print(f"Failed serials    : {len(failed_serials):,}")
print(f"Healthy serials   : {len(healthy_serials):,}")
print(f"Sampled healthy   : {len(sampled_healthy):,}")
print(f"Filtered shape    : {df.shape}")

In [ ]:
NA_VALUES = [
    "", " ", "NA", "N/A", "na", "n/a",
    "NULL", "null", "None", "none",
    "nan", "NaN", "<NA>",
]

df = df.replace(NA_VALUES, np.nan)

str_cols = ["serial_number", "manufacturer", "model"]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

smart_cols = [c for c in df.columns if c.startswith("smart_") and c.endswith("_raw")]
for col in smart_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["failure"] = df["failure"].astype("int8")

df = df.sort_values(["serial_number", "date"]).reset_index(drop=True)

print(df.dtypes)
print("\nTop missing ratios:")
print(df.isna().mean().sort_values(ascending=False).head(10))

## Build SMART Metadata

In [ ]:
d2t_start_time = time.perf_counter()

In [ ]:
threshold_rules = {}
related_map = {}

for _, row in df_smart_des.iterrows():
    name = row['name']
    thresh = row['threshold']
    threshold_rules[name] = thresh
    
    rel_str = row.get('related', '')
    if pd.isna(rel_str) or rel_str in ('—', '', '-'):
        related_list = []
    else:
        related_list = [x.strip() for x in str(rel_str).split(',')]
    related_map[name] = set(related_list)

In [ ]:
pct_attrs = set()
for attr, rule in threshold_rules.items():
    if rule.startswith(('upper_pct:', 'lower_pct:')):
        pct_attrs.add(attr)

quantiles = {}

for col in pct_attrs:
    if col not in df.columns:
        quantiles[col] = None
        continue

    series = pd.to_numeric(df[col], errors='coerce')

    if series.notna().sum() == 0:
        quantiles[col] = None
        continue

    q_vals = series.quantile([0.01, 0.05, 0.10, 0.25, 0.75, 0.90, 0.95, 0.99])

    quantiles[col] = {
        'q01': q_vals.loc[0.01],
        'q05': q_vals.loc[0.05],
        'q10': q_vals.loc[0.10],
        'q25': q_vals.loc[0.25],
        'q75': q_vals.loc[0.75],
        'q90': q_vals.loc[0.90],
        'q95': q_vals.loc[0.95],
        'q99': q_vals.loc[0.99],
    }

## Helper Functions

In [ ]:
def get_description(value, attr_name, threshold_rules, quantiles):
    if pd.isna(value):
        return 'N/A'
    
    rule = threshold_rules.get(attr_name, 'info')
    
    if rule in ('info', 'ignore', 'trend', 'model_specific'):
        return 'N/A'
    
    if rule == 'hard_zero':
        if value == 0:
            return 'normal'
        elif value <= 2:
            return 'high'
        else:
            return 'extremely high'

    if rule == 'soft_zero':
        if value == 0:
            return 'normal'
        elif value <= 5:
            return 'slightly high'
        elif value <= 30:
            return 'high'
        else:
            return 'extremely high'

    if rule.startswith('upper_pct:'):
        try:
            pct = int(rule.split(':', 1)[1])
            q_key = f'q{pct:02d}'
            q_val = quantiles.get(attr_name, {}).get(q_key)
            if q_val is None:
                return 'N/A'
            
            if value > q_val:
                if pct >= 99:
                    return 'extremely high'
                elif pct >= 95:
                    return 'high'
                else:
                    return 'slightly high'
            return 'normal'
        except Exception:
            return 'N/A'
    
    if rule.startswith('lower_pct:'):
        try:
            pct = int(rule.split(':', 1)[1])
            q_key = f'q{pct:02d}'
            q_val = quantiles.get(attr_name, {}).get(q_key)
            if q_val is None:
                return 'N/A'
            
            if value < q_val:
                if pct <= 1 or pct <= 5:
                    return 'extremely low' if pct <= 1 else 'low'
                else:
                    return 'slightly low'
            return 'normal'
        except Exception:
            return 'N/A'
    
    if rule.startswith('absolute:'):
        try:
            limit = float(rule.split(':', 1)[1])
            if value > limit + 15:
                return 'extremely high'
            elif value > limit + 5:
                return 'high'
            elif value > limit:
                return 'slightly high'
            else:
                return 'normal'
        except Exception:
            return 'N/A'
    
    return 'N/A'

In [ ]:
def compute_upcoming_vectorized(group):
    n = len(group)
    if n < 2:
        return ["No"] * n
    
    dates = group['date']
    failures = group['failure']
    
    s = dates.where(failures == 1, pd.NaT)
    next_incl = s.bfill()
    next_strict = next_incl.mask(failures == 1, pd.NaT)
    next_fail_dates = next_strict.bfill()

    has_upcoming = next_fail_dates.notna() & (next_fail_dates <= dates + timedelta(days=7))
    labels = np.full(n, "No", dtype=object)
    yes_mask = has_upcoming.values
    if yes_mask.any():
        yes_dates_str = next_fail_dates[yes_mask].dt.strftime('%Y/%m/%d').values
        yes_labels = np.array([f"Yes (on {d})" for d in yes_dates_str], dtype=object)
        labels[yes_mask] = yes_labels
    
    return labels.tolist()

In [ ]:
def compute_recent_trend_vectorized(
    group,
    attr='smart_195_raw',
    lookback_days=7,
    min_points=3,
    strict_rel_change=0.10,
    general_rel_change=0.03,
    fluctuation_threshold=0.08
):
    n = len(group)
    if n < min_points:
        return ["too few points"] * n

    group = group.sort_values('date').reset_index(drop=True)

    dates = group['date']
    values = group[attr].replace([np.inf, -np.inf], np.nan).astype(float)

    labels = np.full(n, "no data", dtype=object)

    for i in range(n):
        window_start = dates.iloc[i] - timedelta(days=lookback_days)
        past_mask = (dates >= window_start) & (dates <= dates.iloc[i])

        past_vals = values[past_mask & values.notna()]

        if len(past_vals) < min_points:
            labels[i] = "too few points"
            continue

        v_min   = past_vals.min()
        v_max   = past_vals.max()
        v_first = past_vals.iloc[0]
        v_last  = past_vals.iloc[-1]

        if pd.isna(v_first) or pd.isna(v_last):
            labels[i] = "no data"
            continue

        abs_change = v_last - v_first
        range_pct  = (v_max - v_min) / (abs(v_first) + abs(v_last) + 1e-9) / 2   # rough relative range

        if abs_change <= 1e-6 and range_pct < fluctuation_threshold:
            labels[i] = "stable"
            continue

        rel_change = abs_change / (abs(v_first) + 1e-9)

        is_increasing = abs_change > 0
        is_decreasing = abs_change < 0

        is_strict = abs(rel_change) >= strict_rel_change
        is_general = abs(rel_change) >= general_rel_change

        is_fluctuated = range_pct > fluctuation_threshold

        if is_fluctuated:
            labels[i] = "fluctuated"
        elif is_increasing:
            if is_strict:
                labels[i] = "strictly increasing"
            elif is_general:
                labels[i] = "generally increasing"
            else:
                labels[i] = "slightly increasing"
        elif is_decreasing:
            if is_strict:
                labels[i] = "strictly decreasing"
            elif is_general:
                labels[i] = "generally decreasing"
            else:
                labels[i] = "slightly decreasing"
        else:
            labels[i] = "stable"

    return labels.tolist()

## Compute 7-Day Failure Labels

In [ ]:
print("Computing upcoming failure labels...")
df = df.sort_values(['serial_number', 'date']).reset_index(drop=True)

groups = [group for _, group in df.groupby('serial_number')]
num_groups = len(groups)

num_workers = os.cpu_count()
with Pool(num_workers) as pool:
    upcoming_labels = []
    for labels in tqdm(
        pool.imap(compute_upcoming_vectorized, groups),
        total=num_groups,
        desc="Upcoming failure"
    ):
        upcoming_labels.extend(labels)

df['upcoming_failure'] = upcoming_labels

df["failure_7d"] = (
    df["upcoming_failure"].astype(str).str.startswith("Yes")
    | (pd.to_numeric(df["failure"], errors="coerce").fillna(0).astype(int) == 1)
).astype(int)

## Compute Trend Columns

In [ ]:
trendy_attribute_list = df_smart_des.loc[df_smart_des['threshold'] == 'trend', 'name'].tolist()
existing_trendy_attributes = [attr for attr in df.columns if attr.startswith("smart_") and attr in trendy_attribute_list]

In [ ]:
def compute_trend_for_group(group, trendy_col):
    return compute_recent_trend_vectorized(
        group,
        attr=trendy_col,
        lookback_days=7,
        min_points=3,
        strict_rel_change=0.12,
        general_rel_change=0.035,
        fluctuation_threshold=0.085
    )

In [ ]:
trend_columns = []
groups = [g for _, g in df.groupby('serial_number')]

for trendy_col in existing_trendy_attributes:
    def worker(g):
        return compute_trend_for_group(g, trendy_col)
    
    with Pool() as pool:
        results = list(tqdm(
            pool.imap(worker, groups, chunksize=10),
            total=len(groups),
            desc=f"Computing recent trend for {trendy_col}"
        ))

    all_labels = [label for sublist in results for label in sublist]
    trendy_col_name = f"{trendy_col}_trend_last_7d"
    df[trendy_col_name] = all_labels
    trend_columns.append(trendy_col_name)

In [ ]:
feature_columns = [
    col for col in df.columns 
    if col.startswith('smart_') 
    and col in threshold_rules 
    and threshold_rules[col] not in ('info', 'ignore', 'trend', 'model_specific')
]

cols_strong = []
cols_medium = []
for col in feature_columns:
    var_val = df_smart_des.loc[df_smart_des['name'] == col, 'variation']
    if not var_val.empty:
        var = var_val.iloc[0]
        if var == 'Strong':
            cols_strong.append(col)
        elif var == 'Medium':
            cols_medium.append(col)

pod_vault_correlated = set(cols_strong + cols_medium)

## Description Generation

In [ ]:
def generate_single_description(args):
    idx, row, threshold_rules, quantiles, feature_columns, related_map = args
    parts = []

    date_val = pd.to_datetime(row['date'])
    date_str = date_val.strftime('%Y/%m/%d') if pd.notna(date_val) else 'N/A'
    serial = str(row['serial_number']) if pd.notna(row['serial_number']) else 'N/A'
    model = str(row['model']) if pd.notna(row['model']) else 'N/A'
    upcoming = row['upcoming_failure']

    basic_parts = [
        f"Disk: {serial}",
        f"  Date: {date_str}",
        f"  Model: {model}",
        f"  Failure in next 7 days: {upcoming}"
    ]
    parts.extend(basic_parts)


    # --- Find abnormal attributes ---
    abnormal_attrs = set()
    attr_descriptions = {}

    for col in feature_columns:
        val = row[col]
        desc = get_description(val, col, threshold_rules, quantiles)
        attr_descriptions[col] = desc
        if desc not in ('normal', 'N/A'):
            abnormal_attrs.add(col)

    # --- Expand to include related attributes ---
    expanded_attrs = set(abnormal_attrs)
    for attr in abnormal_attrs:
        expanded_attrs.update(related_map.get(attr, set()))
    
    # Keep only valid feature columns
    expanded_attrs = expanded_attrs & set(feature_columns)
    expanded_attrs = sorted(expanded_attrs)  # for consistent order

    if not expanded_attrs:
        parts.append("SMART Features: All normal")
    else:
        parts.append("SMART Features:")
        for col in expanded_attrs:
            val = row[col]
            feature_desc = attr_descriptions.get(col, 'N/A')
            if feature_desc == 'N/A':
                feature_desc = get_description(val, col, threshold_rules, quantiles)

            meta_row = df_smart_des[df_smart_des['name'] == col]
            if meta_row.empty:
                feature_meta_des = col
                desired_trend = 'unknown'
            else:
                meta_row = meta_row.iloc[0]
                feature_meta_des = meta_row.get('description', col)
                desired_trend = meta_row.get('desired', 'unknown')

            line = ""
            if desired_trend == 'lower' or desired_trend == 'higher':
                line = f"{col} | {feature_meta_des} | {desired_trend} the better | current status: {feature_desc}"
            else:
                continue
                
            parts.append(f"  {line}")

    notable_trends = []
    for trend_col in trend_columns:
        trend_val = row[trend_col]
        if trend_val not in ("stable", "too few points", "no data", "fluctuated"):
            base_col = trend_col.replace('_trend_last_7d', '')
            raw_val = row.get(base_col, 'N/A')
            meta_row = df_smart_des[df_smart_des['name'] == base_col]
            if meta_row.empty:
                feature_meta_des = col
            else:
                meta_row = meta_row.iloc[0]
                feature_meta_des = meta_row.get('description', col)
            line = f"{base_col} | {feature_meta_des} | 7-Day Trend: {trend_val} "
            parts.append(f"  {line}")

    return idx, '\n'.join(parts)

In [ ]:
def init_worker(threshold_rules, quantiles, feature_columns, related_map):
    global threshold_rules_g, quantiles_g, feature_columns_g, related_map_g
    threshold_rules_g = threshold_rules
    quantiles_g = quantiles
    feature_columns_g = feature_columns
    related_map_g = related_map


def generate_wrapper(pos_and_row):
    pos, row_dict = pos_and_row
    full_args = (pos, row_dict, threshold_rules_g, quantiles_g, feature_columns_g, related_map_g)
    return generate_single_description(full_args)

## Parallel processing

In [ ]:
batch_size = 40000
num_workers = multiprocessing.cpu_count()
output_dir = OUTPUT_TEXTUAL_DIR
output_dir.mkdir(parents=True, exist_ok=True)

saved_files = []
current_start = df["date"].min().to_period("M").start_time
max_date = df["date"].max()

overall_pbar = tqdm(total=len(df), desc="Overall progress", unit="rows")

with ProcessPoolExecutor(
    max_workers=num_workers,
    initializer=init_worker,
    initargs=(threshold_rules, quantiles, feature_columns, related_map),
) as executor:
    while current_start <= max_date:
        month_start = current_start.to_period("M").start_time
        month_end = current_start.to_period("M").end_time

        mask = (df["date"] >= month_start) & (df["date"] <= month_end)
        batch_df = df[mask].copy()

        if batch_df.empty:
            current_start = month_end + pd.Timedelta(days=1)
            continue

        chunk_start_str = month_start.strftime("%Y-%m-%d")
        chunk_end_str = month_end.strftime("%Y-%m-%d")

        chunk_file = output_dir / f"alibaba_disk_textual_{chunk_start_str}_to_{chunk_end_str}.csv"

        print(
            f"Processing chunk: {chunk_start_str} to {chunk_end_str} "
            f"({len(batch_df)} rows) -> {chunk_file}"
        )

        records = batch_df.to_dict(orient="records")
        chunk_descriptions = ["Generation failed"] * len(batch_df)

        chunk_pbar = tqdm(total=len(batch_df), desc="Generating descriptions", leave=False)

        for local_start in range(0, len(batch_df), batch_size):
            local_end = min(local_start + batch_size, len(batch_df))
            sub_records = records[local_start:local_end]

            futures = []
            for sub_pos, row_dict in enumerate(sub_records):
                pos = local_start + sub_pos
                future = executor.submit(generate_wrapper, (pos, row_dict))
                futures.append(future)

            for future in as_completed(futures):
                try:
                    idx, desc = future.result()
                    chunk_descriptions[idx] = desc
                except Exception as exc:
                    print(f"Warning: generation error: {exc}")
                chunk_pbar.update(1)
                overall_pbar.update(1)

        chunk_pbar.close()

        new_chunk_df = pd.DataFrame({"description": chunk_descriptions})
        new_chunk_df["failure"] = batch_df["failure"].values
        new_chunk_df["failure_7d"] = batch_df["failure_7d"].values
        new_chunk_df["date"] = batch_df["date"].values
        new_chunk_df["serial_number"] = batch_df["serial_number"].values
        new_chunk_df["model"] = batch_df["model"].values

        new_chunk_df.to_csv(chunk_file, index=False)

        successful_count = (new_chunk_df["description"] != "Generation failed").sum()
        print(f"Saved {len(new_chunk_df)} rows ({successful_count} successful) to {chunk_file}")

        saved_files.append(chunk_file)

        del batch_df, records, chunk_descriptions, new_chunk_df
        gc.collect()

        current_start = month_end + pd.Timedelta(days=1)

overall_pbar.close()
print("All chunks processed.")

In [ ]:
d2t_end_time = time.perf_counter()
elapsed_time = d2t_end_time - d2t_start_time
print(f"D2T elapsed time: {elapsed_time:.6f} seconds")